# 1+2

This notebook merges the two DFs from A1 and A2. A .fasta will be created as well, which will be the foundation for the DB used in the pipeline.
From this, one can deduce a .tsv with all reactions in both CHEBI and Name format for each transporter. The key output is the reactions for each identified protein in the BLASTp from the pipeline.

In [47]:
import pandas as pd
import requests
from Bio import SeqIO
from io import StringIO
import re
import numpy as np

In [48]:
df1 = pd.read_csv("../Approach 1/transporters1_df.tsv", sep="\t")
df2 = pd.read_csv("../Approach 2/transporters2_df.tsv", sep="\t")
df = pd.concat([df1, df2], ignore_index=True)

Noticing that some rows have missing TCIDs for known AAs that are present in TCDB. Therfore, a mapping of TCIDs are performed.

In [49]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def parse_data(substrates_txt, aa_txt):

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
                        for line in substrates_lines
                        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])

    # AA sequence (TCID, UID and AA)
    fasta_io = StringIO(aa_txt)
    tc_data = [[record.description.split("|")[3].split()[0],  # TCID
                record.description.split("|")[2],  # UID
                str(record.seq)]  # AA
                for record in SeqIO.parse(fasta_io, "fasta")]
    
    df_aa = pd.DataFrame(tc_data, columns=["TCID", "UID", "AA"])

    return df_substrates, df_aa

tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"
tc_aa_url = "https://www.tcdb.org/public/tcdb"

tc_substrates_txt = fetch_data(tc_substrates_url)
tc_aa_txt = fetch_data(tc_aa_url)


df_substrates, df_aa = parse_data(tc_substrates_txt, tc_aa_txt)

In [50]:
df_aa_unique = df_aa.drop_duplicates(subset="AA")
df.loc[df["TCID"].isna(), "TCID"] = df["AA"].map(df_aa_unique.set_index("AA")["TCID"])

Now, from these newly obtained TCIDs, one also needs to extract their ChEBI information, family, mechanism and so on. Whatever is retrievable from TCDB.

In [51]:
df_updated = df.merge(df_substrates, on="TCID", how="left", suffixes=("", "_new"))

df_updated["CHEBI ID"] = df_updated["CHEBI ID"].fillna(df_updated["CHEBI ID_new"])
df_updated["CHEBI Name"] = df_updated["CHEBI Name"].fillna(df_updated["CHEBI Name_new"])

df_updated = df_updated.drop(columns=["CHEBI ID_new", "CHEBI Name_new"])

# Need to do s2p on the column again. The same goes for p2n...
df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
df_p2n = pd.read_csv("../ChEBI/p2n.tsv", sep="\t")

secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
primary_to_name = dict(zip(df_p2n["Primary_ID"], df_p2n["ChEBI Name"]))

df_updated["CHEBI ID"] = df_updated["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))
df_updated["CHEBI Name"] = df_updated.apply(lambda row: primary_to_name.get(row["CHEBI ID"], row["CHEBI Name"]), axis=1)

df_updated = df_updated.drop_duplicates()

The next step is to extract all families from the previous Nan-TCIDs, and then their mechanisms, acting entities and reactions.

In [52]:
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_final.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "⇌", "â†’": "→"}, regex=True)

# Creating "Family" only for Nan-values
df_updated.loc[df_updated["Family"].isna(), "Family"] = df_updated.loc[df_updated["Family"].isna(), "TCID"].apply(
    lambda x: ".".join(str(x).split(".")[:3]) if pd.notna(x) else x)

df_updated = df_updated.merge( df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left",suffixes=("", "_new"))

df_updated["Mechanism"] = df_updated["Mechanism"].fillna(df_updated["Mechanism_new"])
df_updated["Acting Entity"] = df_updated["Acting Entity"].fillna(df_updated["Acting Entity_new"])

df_updated = df_updated.drop(columns=["Mechanism_new", "Acting Entity_new"])

def create_reaction_row(row):
    if pd.notna(row["Reaction"]):
        return row["Reaction"]

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    
    mechanisms = str(row["Mechanism"]).split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = [mechanism.replace(entity, chebi_name) for mechanism, entity in zip(mechanisms, acting_entities)]
    
    return ", ".join(reactions)

df_updated["Reaction"] = df_updated.apply(create_reaction_row, axis=1)

The next step is to implement the actual accession ID used by TCDB. The UID column MOSTLY correspond to the given AA, but sometimes, they have been RSIDs. UID is the final column after all RSIDs have been converted to UIDs, in order to map between TCDB-UniProt-Rhea, both ways. For referencing purposes towards the transporters, AAs and TCIDs in TCDB, a new column is added. That is the AID (Accession ID), which is the ID used by TCDB for any TCID (or a protein in a larger complex that all have the same TCID). Finally, just some minor readjustments for the columns, making it just a tad easier to look at, before the final DF is obtained.

In [53]:
df_temp = df_updated.drop_duplicates().reset_index(drop=True)
df_temp = df_temp.merge(df_aa[["AA", "UID"]], on="AA", how="left", suffixes=("", "_new"))
df_temp.rename(columns={"UID_new":"AID"}, inplace=True)
cols = ["AID"] + [col for col in df_temp.columns if col != "AID"]
df_temp = df_temp[cols]

Now, a couple of issues have appeared. They need to be tackled by addressing the creating of the reference-DB in order to make it generalizable.

1)  The first step is to create a column with the reactions in ChEBI format. (CHEBI:1234 + CHEBI:56789 = CHEBI:1234 + CHEBI:56789)\
This must be done for both TCDB and RHEA.
2) Then, these reactions will be gathered in a common column for reactions related to that specific transporter. This will compress the final output (speaking in amount of lines). But this will be done for a new df, and saved as another file, in order to maintain as much info as possible.

Below is the implementation of step 1.

In [ ]:
df_eq = df_temp.copy()

name2chebi_dict = {}

with open("../ChEBI/name2chebi.txt", "r") as file:
    for line in file:
        parts = line.strip().split(" ")
        name, chebi_id = parts
        name2chebi_dict[name] = chebi_id

def tcdb_convert_to_chebi(row):
    reaction = row["Reaction"]
    
    if pd.notna(reaction) and isinstance(reaction, str):

        reaction = re.sub(r"\s*\(in\)|\(out\)", "", reaction)

        chebi_name = row["CHEBI Name"]
        chebi_id = row["CHEBI ID"]
        
        if pd.notna(chebi_name) and pd.notna(chebi_id):
            reaction = reaction.replace(chebi_name, chebi_id)

        for name, chebi_id in name2chebi_dict.items():
            reaction = reaction.replace(name, chebi_id)
            
        # Remove any double spaces
        reaction = re.sub(r"\s{2,}", " ", reaction).strip()
    return reaction


def rhea_convert_to_chebi(row):
    equation = str(row["R:Equation"]) if pd.notnull(row["R:Equation"]) else ""
    chebi_str = str(row["R:ChEBI identifier"]) if pd.notnull(row["R:ChEBI identifier"]) else ""

    # Avoid errors when empty
    if not equation.strip() or not chebi_str.strip():
        return np.nan

    chebis = [c.strip() for c in chebi_str.split(";")]

    # Clean equation text
    cleaned = re.sub(r"\((in|out)\)", "", equation)
    cleaned = re.sub(r"\(n\+1\)", "", cleaned)
    cleaned = re.sub(r"\(n\)", "", cleaned)
    cleaned = re.sub(r"\s+n\s+", " ", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()

    # Split on " + " and " = ", preserving operators
    tokens = re.split(r" ([+=]) ", cleaned)
    parts = [t.strip() for t in tokens if t.strip()]

    processed_names = []
    processed_ops = []

    i = 0
    while i < len(parts):
        token = parts[i]
        if token in ["+", "="]:
            processed_ops.append(f" {token} ")
        else:
            match = re.match(r"^(\d+)\s+(.+)$", token)
            if match:
                stoich, name = match.groups()
                processed_names.append((name, stoich))
            else:
                processed_names.append((token, ""))  # no stoichiometry
        i += 1

    # Map names to CHEBI IDs
    name_to_chebi = {}
    chebi_index = 0
    chebi_result = []
    
    for name, stoich in processed_names:
        if name not in name_to_chebi:
            if chebi_index < len(chebis):
                name_to_chebi[name] = chebis[chebi_index]
                chebi_index += 1
            else:
                name_to_chebi[name] = "MISSING_CHEBI" # Not an issue now, but kept in for later iterations
        chebi_id = name_to_chebi[name]
        chebi_result.append(f"{stoich} {chebi_id}".strip())

    # Reconstruct final reaction string
    result = [chebi_result[0]]
    for i, op in enumerate(processed_ops):
        result.append(op)
        result.append(chebi_result[i + 1])

    # Remove any double spaces
    result = "".join(result)
    result = re.sub(r"\s{2,}", " ", result).strip()
    return result


# Apply the functions
df_eq["Rhea:Reaction:CHEBI"] = df_eq.apply(rhea_convert_to_chebi, axis=1)
df_eq["TCDB:Reaction:CHEBI"] = df_eq.apply(tcdb_convert_to_chebi, axis=1)

Now, a separate DF is created to more accurately depict what reactions appear for each transporter. This will potentially be of use to determine final reactions appearing after BLASTp.

In [55]:
df_reactions = df_eq.copy()
df_reactions.drop(["UID", "CHEBI ID", "CHEBI Name", "Family", "Mechanism",
                "Acting Entity", "RID", "R:ChEBI name", "R:ChEBI identifier"], axis=1, inplace=True)
df_reactions = df_reactions.groupby(["AID", "TCID", "AA"], as_index=False).agg(lambda x: list(x))

cols = ["Reaction", "R:Equation", "Rhea:Reaction:CHEBI", "TCDB:Reaction:CHEBI"]

for col in cols:
    df_reactions[col] = df_reactions[col].apply(lambda x: list(dict.fromkeys(x)) if isinstance(x, list) else x)

tcdb_rhea_reactions is a DF containing proteins where there are reactions from both TCDB and Rhea. The purpose is to gain biological insight from this, and see what kind of reactions are maintained here. What is common? This work will continue in another notebook.\
filtered_reactions is the DF that is used to present all reactions related to the proteins found in the pipeline.

In [56]:
def not_nan_list(x):
    return not (isinstance(x, list) and len(x) == 1 and pd.isna(x[0]))

tcdb_rhea_reactions = df_reactions[
    df_reactions["Rhea:Reaction:CHEBI"].apply(not_nan_list) &
    df_reactions["TCDB:Reaction:CHEBI"].apply(not_nan_list)
]

filtered_reactions = df_reactions[
    df_reactions["Rhea:Reaction:CHEBI"].apply(not_nan_list) |
    df_reactions["TCDB:Reaction:CHEBI"].apply(not_nan_list)
]

tcdb_rhea_reactions.to_csv("tcdb_rhea_common_reactions.tsv", sep="\t", index=False)

Below, (in/out) are removed, as are the duplicates that then follow. Only after this, it is possible to line up reactions with their repsective names and CHEBI IDs correctly. Subsequently, rows that dont match number of reactions in name and CHEBI columns are deleted (<10 instances).

In [57]:
def normalize_reaction_list(reaction_list):
    if isinstance(reaction_list, list):
        if reaction_list == [np.nan]:
            return [np.nan]
        return list(dict.fromkeys([
            r.replace("(in)", "").replace("(out)", "").rstrip()
            for r in reaction_list if isinstance(r, str)
        ]))
    return []

filtered_reactions = filtered_reactions.copy()
filtered_reactions.loc[:, "R:Equation"] = filtered_reactions["R:Equation"].apply(normalize_reaction_list)
filtered_reactions.loc[:, "Reaction"] = filtered_reactions["Reaction"].apply(normalize_reaction_list)
filtered_reactions.loc[:, "Rhea:Reaction:CHEBI"] = filtered_reactions["Rhea:Reaction:CHEBI"].apply(normalize_reaction_list)
filtered_reactions.loc[:, "TCDB:Reaction:CHEBI"] = filtered_reactions["TCDB:Reaction:CHEBI"].apply(normalize_reaction_list)

filtered_reactions.loc[:,"Rhea_match"] = filtered_reactions.apply(
    lambda row: len(row["Rhea:Reaction:CHEBI"]) == len(row["R:Equation"]) 
    if isinstance(row["Rhea:Reaction:CHEBI"], list) and isinstance(row["R:Equation"], list)
    else False,
    axis=1
)

filtered_reactions.loc[:, "TCDB_match"] = filtered_reactions.apply(
    lambda row: len(row["TCDB:Reaction:CHEBI"]) == len(row["Reaction"]) 
    if isinstance(row["TCDB:Reaction:CHEBI"], list) and isinstance(row["Reaction"], list)
    else False,
    axis=1
)

filtered_reactions = filtered_reactions[(filtered_reactions["Rhea_match"]) & (filtered_reactions["TCDB_match"])]

filtered_reactions.drop(columns=["Rhea_match", "TCDB_match"], inplace=True)
filtered_reactions.rename(columns={"Reaction": "TCDB:Reaction:Name","R:Equation": "Rhea:Reaction:Name"}, inplace=True)
filtered_reactions.reset_index(drop=True, inplace=True)

Now, the merging of Rhea and TCDB reactions will be done. It is important to maintain correct mapping between CHEBI and Name. Therefore, it is fruitful to do this with a dictionary, where the key is the CHEBI reaction, and the value is the name reaction. This ensures that if a reaction has identical CHEBIs, only one CHEBI is saved. Also, it is only worth keeping the first value, which will be Rhea, as this is more "normalized" than the TCDB convention.

In [58]:
def merge_reactions(row):
    merged_dict = {}

    # Start with Rhea
    rhea_chebis = row.get("Rhea:Reaction:CHEBI", [])
    rhea_names = row.get("Rhea:Reaction:Name", [])
    if isinstance(rhea_chebis, list) and isinstance(rhea_names, list):
        for chebi, name in zip(rhea_chebis, rhea_names):
            if isinstance(chebi, str) and isinstance(name, str):
                merged_dict[chebi] = name

    # Add from TCDB only if CHEBI not already in dict
    tcdb_chebis = row.get("TCDB:Reaction:CHEBI", [])
    tcdb_names = row.get("TCDB:Reaction:Name", [])
    if isinstance(tcdb_chebis, list) and isinstance(tcdb_names, list):
        for chebi, name in zip(tcdb_chebis, tcdb_names):
            if isinstance(chebi, str) and isinstance(name, str) and chebi not in merged_dict:
                merged_dict[chebi] = name

    # Return keys and values as lists
    return pd.Series({
        "Reaction:CHEBI": list(merged_dict.keys()),
        "Reaction:Name": list(merged_dict.values())
    })

# Apply to each row, join columns into df, drop columns not needed anymore
merged = filtered_reactions.apply(merge_reactions, axis=1)
filtered_reactions = pd.concat([filtered_reactions, merged], axis=1)
filtered_reactions.drop(columns=["TCDB:Reaction:Name", "Rhea:Reaction:Name", "Rhea:Reaction:CHEBI", "TCDB:Reaction:CHEBI"], inplace=True)

# Replace ⇌ and → with = as it has snuck into the equations somehow
def replace_chemical_reactions(row):
    if isinstance(row, list):
        return [item.replace("⇌", "=").replace("→", "=") if isinstance(item, str) else item for item in row]
    return row

filtered_reactions["Reaction:Name"] = filtered_reactions["Reaction:Name"].apply(replace_chemical_reactions)
filtered_reactions["Reaction:CHEBI"] = filtered_reactions["Reaction:CHEBI"].apply(replace_chemical_reactions)

filtered_reactions

,AID,TCID,AA,Reaction:CHEBI,Reaction:Name
0,5IIP_A,3.A.1.12.16,GSSHHHHHHSSGLVPRGSHMASGQNRLIQDRPNDKTVEGVMIKPIT...,[CHEBI:15354 + CHEBI:15422 = CHEBI:15354 + CHE...,"[choline + ATP = choline + ADP + Pi, carniti..."
1,5O8F_E,1.A.9.5.14,ETGQSVNDPGNMSFVKETVDKLLKGYDIRLRPDFGGPPVCVGMNID...,"[CHEBI:17996 = CHEBI:17996, CHEBI:17544 = CHEB...","[chloride = chloride, hydrogencarbonate = hy..."
2,6LH8_A,1.C.4.8.1,MSASVTVLWDKEIEGSNEVVKVDEMVASNISNVKVEFYLKERHFDR...,[CHEBI:25367 = CHEBI:25367],[molecule = molecule]
3,7TJ9_A,1.A.1.10.22,MWSHPQFEKGGGSGGGSGGSAWSHPQFEKFFSFFDYKDDDDKGGSG...,[CHEBI:29101 = CHEBI:29101],[sodium(1+) = sodium(1+)]
4,7Z8E_A,3.A.1.5.49,MGSSHHHHHHSSGLVPRGSHMEEQPVWHHATSSIGEPKYKDGFARF...,[CHEBI:16670 + CHEBI:15422 = CHEBI:16670 + CHE...,[peptide + ATP = peptide + ADP + Pi]
...,...,...,...,...,...
8661,h9ub92,1.A.77.3.16,MSNSPSNDVLRFVVRIIIYSLVVSSVTLLLGVLLRRFSFALGVLIG...,[CHEBI:3473 = CHEBI:3473],[cation = cation]
8662,o34814,3.A.1.140.8,MIEMKEVYKAYPNGVKALNGISVTIHPGEFVYVVGPSGAGKSTFIK...,[solute + CHEBI:15422 = solute + CHEBI:16761 +...,"[solute + ATP = solute + ADP + Pi, substrate..."
8663,o84418,1.B.12.1.8,MSSMKWLSATAVFAAVLPSVSGFCFPEPKELNFSRVGTSSSTTFTE...,[Protein virulence factor (periplasm) = Protei...,[Protein virulence factor (periplasm) = Protei...
8664,s6ex81,2.A.3.3.23,MGFMRKADFELYRDADKHYNQVLTTRDFLALGVGTIISTSIFTLPG...,[CHEBI:17191 + n CHEBI:15378 = CHEBI:17191 + n...,"[L-isoleucine + n H+ = L-isoleucine + n H+,..."


In order to take into account that a protein might be a complex, conisisting of subunits, the AIDs need to come into use once more. One subprotien might me mapped to a certain set of reactions, but it only happens when the whole complex appears. In order to keep as much info as possible, it is reasonable to map these reactions to all other subproteins that is part of the same complex. Therefore, a comparison of df_eq and filtered_reactions is needed. df_eq contains all the transporters and AAs even without reactions. From there, one can obtain the other TCIDs, and map these to the given AID, then include the reactions, before finally identifying AAs without any reactions.

In [59]:
# Right merge to keep all AIDs from df_eq
merged_df = pd.merge(
    filtered_reactions[["AID", "TCID", "AA", "Reaction:Name", "Reaction:CHEBI"]],
    df_eq[["AID", "TCID", "AA"]],
    on="AID",
    how="right",
    suffixes=("_filtered", "_eq")
)

merged_df.drop_duplicates(subset=["AID", "TCID_eq", "AA_eq"], keep="first", inplace=True)
merged_df["TCID_filtered"] = merged_df["TCID_filtered"].fillna(merged_df["TCID_eq"])
merged_df["AA_filtered"] = merged_df["AA_filtered"].fillna(merged_df["AA_eq"])
merged_df.drop(columns=["TCID_eq", "AA_eq"], inplace=True)
merged_df.rename(columns={"TCID_filtered": "TCID","AA_filtered": "AA"}, inplace=True)

In [60]:
def merge_reactions_within_tcid(group):
    # Combine and deduplicate while preserving order
    names = []
    chebis = []

    if group["Reaction:Name"].isna().all() or group["Reaction:CHEBI"].isna().all():
        group["Reaction:Name"] = np.nan
        group["Reaction:CHEBI"] = np.nan

    else:
        for name_list, chebi_list in zip(group["Reaction:Name"], group["Reaction:CHEBI"]):
            if isinstance(name_list, list) and isinstance(chebi_list, list):
                for n, c in zip(name_list, chebi_list):
                    if n not in names:
                        names.append(n)
                        chebis.append(c)
        
        chebi_to_name = {chebis[i]: names[i] for i in range(len(chebis))}
        merged_chebis = list(chebi_to_name.keys())
        merged_names = [chebi_to_name[c] for c in merged_chebis]

        group["Reaction:Name"] = [merged_names] * len(group)
        group["Reaction:CHEBI"] = [merged_chebis] * len(group)

    return group

merged_df["TCID_cp"] = merged_df["TCID"]
merged_df = merged_df.groupby("TCID").apply(lambda group: merge_reactions_within_tcid(group), include_groups=False).reset_index(drop=True)
merged_df.rename(columns={"TCID_cp":"TCID"}, inplace=True)
merged_df = merged_df[["AID", "TCID", "AA", "Reaction:Name", "Reaction:CHEBI"]]


def fill_missing_reaction(cell):
    return "no_reaction_identified" if isinstance(cell, float) and np.isnan(cell) else cell

merged_df["Reaction:Name"] = merged_df["Reaction:Name"].apply(fill_missing_reaction)
merged_df["Reaction:CHEBI"] = merged_df["Reaction:CHEBI"].apply(fill_missing_reaction)
df_fasta = merged_df.drop_duplicates(subset=["AID", "TCID", "AA"])

df_fasta

,AID,TCID,AA,Reaction:Name,Reaction:CHEBI
0,P0A334,1.A.1.1.1,MPPMLSGLLARLVKLLLGRHGSALHWRAAGAATVLLVIVLLAGSYL...,"[potassium(1+) = potassium(1+), water = water]","[CHEBI:29103 = CHEBI:29103, CHEBI:15377 = CHEB..."
1,P08104,1.A.1.10.1,MAQALLVPPGPESFRLFTRESLAAIEKRAAEEKAKKPKKEQDIDDE...,[Na(+) = Na(+)],[CHEBI:29101 = CHEBI:29101]
2,O01307,1.A.1.10.10,MSDDSSSISEEERSLFRPFTRESLAAIEARIAEEYAKQKELEKKRA...,[sodium(1+) = sodium(1+)],[CHEBI:29101 = CHEBI:29101]
3,Q86D77,1.A.1.10.11,MPPAPAETALSANTEQPAFSTSSATPHALALPIAEDGVHADHDDDD...,[sodium(1+) = sodium(1+)],[CHEBI:29101 = CHEBI:29101]
4,Q99250,1.A.1.10.12,MAQSVLVPPGPDSFRFFTRESLAAIEQRIAEEKAKRPKQERKDEDD...,[Na(+) = Na(+)],[CHEBI:29101 = CHEBI:29101]
...,...,...,...,...,...
24192,W7DA51,9.B.98.1.6,MKIERKELVKFTVIILIFNIVLVFLTVLFFKRMDSNIPLATPAYGV...,no_reaction_identified,no_reaction_identified
24193,M0LLS9,9.B.98.1.7,MNGGPNGSRDDAEDDRDGTGDGTGDDSAGRFEFGPEREPVDGSSTD...,no_reaction_identified,no_reaction_identified
24194,C7T873,9.B.98.1.8,MKKRLTYHLDVIGHYLLVGWLLFIAITIMVSLLTYIVMDANPAFIH...,no_reaction_identified,no_reaction_identified
24195,WP_138772272.1,9.B.98.1.9,MRQSQFEQRYRPLWERLEATLKTLEKTRRPGDAAADFAADYQSLCH...,no_reaction_identified,no_reaction_identified


The last step is to write the DFs to .tsv and .fasta. The subset of identical rows for AID, TCID and AA is quite substantial, so the final .fasta is not very large (approx. 24200 transporters). But these will further be mapped to several reactions, substrates and mechanisms further down the pipeline that will come. This reduction of the subset is done to reduce BLAST-time.

In [61]:
fasta_file = "transporters.fasta"

with open(fasta_file, "w") as f:
    for _, row in df_fasta.iterrows():
        aid = row["AID"]
        tcid = row["TCID"]
        sequence = row["AA"]
        f.write(f">{aid}|{tcid}\n{sequence}\n")

df_fasta.to_csv("transporters_df.tsv", sep="\t", index=False)